# Knowledge-Based Movie Recommender

This project implements a movie recommendation system that combines movie metadata, Prolog-based knowledge representation, content similarity rules, and user ratings.

The workflow includes:
- construction of a movie knowledge base from metadata,
- definition of similarity rules in Prolog,
- content-based recommendation,
- user-preference modelling from ratings,
- evaluation using Precision, Recall, and F1-score.

## Environment Setup

The project uses Python together with SWI-Prolog through the `pyswip` interface.

Required data files are loaded directly from the repository directory.

### Dependencies

This project requires:

- Python 3
- SWI-Prolog
- PySwip
- pandas
- NumPy
- scikit-learn
- tqdm

SWI-Prolog must be installed on the system before using the Python–Prolog interface through PySwip.

## Knowledge Base Construction

In [1]:
from pyswip import Prolog

In [2]:
import pandas as pd

# Load movie metadata from the repository directory
data = pd.read_csv("movies_metadata.csv")

# Replace missing values with a placeholder
data = data.fillna("UNK")

data.head()

,Unnamed: 0,budget,genres,homepage,id,plot_keywords,language,original_title,overview,popularity,...,tagline,movie_title,vote_average,num_voted_users,title_year,country,director_name,actor_1_name,actor_2_name,actor_3_name
0,0,237000000,Action|Adventure|Fantasy|Science Fiction,http://www.avatarmovie.com/,19995,culture clash|future|space war|space colony|so...,English,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,...,Enter the World of Pandora.,Avatar,7.2,11800,2009.0,United States of America,James Cameron,Zoe Saldana,Sigourney Weaver,Stephen Lang
1,1,300000000,Adventure|Fantasy|Action,http://disney.go.com/disneypictures/pirates/,285,ocean|drug abuse|exotic island|east india trad...,English,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,...,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,2007.0,United States of America,Gore Verbinski,Orlando Bloom,Keira Knightley,Stellan Skarsgård
2,2,245000000,Action|Adventure|Crime,http://www.sonypictures.com/movies/spectre/,206647,spy|based on novel|secret agent|sequel|mi6|bri...,Français,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,...,A Plan No One Escapes,Spectre,6.3,4466,2015.0,United Kingdom,Sam Mendes,Christoph Waltz,Léa Seydoux,Ralph Fiennes
3,3,250000000,Action|Crime|Drama|Thriller,http://www.thedarkknightrises.com/,49026,dc comics|crime fighter|terrorist|secret ident...,English,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,...,The Legend Ends,The Dark Knight Rises,7.6,9106,2012.0,United States of America,Christopher Nolan,Michael Caine,Gary Oldman,Anne Hathaway
4,4,260000000,Action|Adventure|Science Fiction,http://movies.disney.com/john-carter,49529,based on novel|mars|medallion|space travel|pri...,English,John Carter,"John Carter is a war-weary, former military ca...",43.926995,...,"Lost in our world, found in another.",John Carter,6.1,2124,2012.0,United States of America,Andrew Stanton,Lynn Collins,Samantha Morton,Willem Dafoe


In [3]:
def clean_text(text):
  text = text.replace(u'\xa0', u'')
  text = text.replace(u"'", u'')
  return text

In [4]:
from pyswip import Prolog
# Initialize the Prolog knowledge base
prolog = Prolog()

# Convert each movie record into Prolog facts
literals = []
movie_score = {}
for row in data.itertuples(index=True, name='Pandas'):

  MovieTitle = clean_text(getattr(row, 'movie_title'))
  budget = getattr(row, 'budget')
  literals.append("budget('"+ MovieTitle +"','" + str(budget) +"')")

  for genre in getattr(row, 'genres').split("|"):
    literals.append("genre('"+ MovieTitle +"','"+ genre +"')")

  homepage = clean_text(getattr(row, 'homepage'))
  literals.append("homepage('"+ MovieTitle +"','" + homepage +"')")

  id = getattr(row, 'id')
  literals.append("id('"+ MovieTitle +"','" + str(id) +"')")

  for plot_keywords in getattr(row, 'plot_keywords').split("|"):
   PlotKeywords= clean_text(plot_keywords)
   literals.append("plot_keywords('"+MovieTitle +"', '"+ PlotKeywords+"')")

  language = getattr(row, 'language')
  literals.append("language('"+ MovieTitle +"','" + language +"')")

  for original_title in clean_text(getattr(row, 'original_title')).split("|"):
   literals.append("original_title('"+MovieTitle +"', '"+ original_title+"')")

  overview= clean_text(getattr(row, 'overview'))
  literals.append("overview('"+MovieTitle + "','" + overview +"')")

  popularity = getattr(row, 'popularity')
  literals.append("popularity('"+ MovieTitle +"','" + str(popularity) +"')")

  for production_companies in list(eval(getattr(row, 'production_companies'))):
    production_companies = production_companies["name"].replace("'",'')
    literals.append("production_company('"+MovieTitle + "', '" + production_companies +"')")

  for production_countries in list(eval(getattr(row, 'production_countries'))):
    production_countries = clean_text(production_countries["name"])
    literals.append("production_country('"+MovieTitle + "', '" + production_countries +"')")

  release_date = getattr(row, 'release_date')
  literals.append("release_date('"+ MovieTitle +"','" + str(release_date) +"')")

  gross = getattr(row, 'gross')
  literals.append("gross('"+ MovieTitle +"','" + str(gross) +"')")

  duration = getattr(row, 'duration')
  literals.append("duration('"+ MovieTitle +"','" + str(duration) +"')")

  for spoken_languages in list(eval(getattr(row, 'spoken_languages'))):
    spoken_languages = clean_text(spoken_languages["name"])
    literals.append("spoken_languages('"+MovieTitle + "', '" + spoken_languages +"')")

  status = getattr(row, 'status')
  literals.append("status('"+ MovieTitle +"','" + status +"')")

  tagline= clean_text(getattr(row, 'tagline'))
  literals.append("tagline('"+MovieTitle + "','" + tagline +"')")

  vote_average = getattr(row, 'vote_average')
  literals.append("vote_average('"+ MovieTitle +"','" + str(vote_average) +"')")

  num_voted_users = getattr(row, 'num_voted_users')
  literals.append("num_voted_users('"+ MovieTitle +"','" + str(num_voted_users) +"')")

  if(getattr(row, 'title_year')!="UNK"):
   title_year = int(getattr(row, 'title_year'))
   literals.append("title_year('"+ MovieTitle +"','" + str(title_year) +"')")

  if(getattr(row, 'title_year')!="UNK"):
   year = int(getattr(row, 'title_year'))
   decade = str(year - year % 10)
   literals.append("decade('"+ MovieTitle +"','" + decade +"')")

  country = getattr(row, 'country')
  literals.append("country('"+ MovieTitle +"','" + country +"')")

  director_name = clean_text(getattr(row, 'director_name'))
  literals.append("director_name('"+ MovieTitle +"','" + director_name +"')")

  actor_1_name = clean_text(getattr(row, 'actor_1_name'))
  literals.append("actor_name('"+ MovieTitle +"','" + actor_1_name +"')")

  actor_2_name = clean_text(getattr(row, 'actor_2_name'))
  literals.append("actor_name('"+ MovieTitle +"','" + actor_2_name +"')")

  actor_3_name = clean_text(getattr(row, 'actor_3_name'))
  literals.append("actor_name('"+ MovieTitle +"','" + actor_3_name +"')")


# Sort Prolog facts for consistent ordering
literals.sort()

# Save the generated knowledge base locally
with open("literals.txt", "w", encoding="utf-8") as file:
    for literal in literals:
        file.write(literal + "\n")

# Add generated facts to the Prolog knowledge base
for literal in literals:
    prolog.assertz(literal)

# Load the predefined Prolog rules
prolog.consult("db.pl")

### Similarity Rules

Movie similarity is represented through Prolog rules derived from shared movie attributes. The current knowledge base identifies content similarity based on common genre information, which is then used by the recommendation system.

## Content-Based Recommendation

Movie recommendations are generated using the similarity rules defined in the Prolog knowledge base.

The recommender identifies movies with shared characteristics, with higher similarity levels representing stronger matches.


In [10]:
def simple_recommender(movie):
    s = set()
    q = prolog.query("find_sim_1('" + movie +"',M)")
    for soln in q:
        m = soln['M']
        if m not in s:
            s.add(soln['M'])
    q.close()
    answers = s
    return answers

### Recommendation Examples

The following examples demonstrate recommendations generated for selected movies using the content-based similarity rules.

In [11]:
l=[]
l.append(simple_recommender("Avatar"))
print(l)

[{'Stripes', 'Space Cowboys', 'The Last Dragon', 'The Andromeda Strain', 'Alive', 'White Squall', 'Miss Congeniality 2: Armed and Fabulous', 'The Abyss', 'Tom Jones', 'Zoom', 'VeggieTales: The Pirates Who Dont Do Anything', 'Risen', 'The X Files: I Want to Believe', 'Lost in Space', 'Queen Crab', 'Windtalkers', 'Gangster Squad', 'Firestarter', 'Ice Age: The Meltdown', 'The Proposition', 'Tremors', 'Proof of Life', 'El Mariachi', 'Pandorum', 'Warlock', 'The Other Side of Heaven', 'Mad Max 2: The Road Warrior', 'Tango & Cash', 'The Incredibles', 'Nerve', 'Solaris', 'Red Sonja', 'The Boxtrolls', 'Chappie', 'The Hunger Games', 'Agora', 'Two Brothers', 'Down Terrace', 'Shooter', 'Training Day', 'Crossroads', 'Exit Wounds', 'Run All Night', 'Hollow Man', 'Steel', 'The Core', 'Night at the Museum: Battle of the Smithsonian', 'The Life Aquatic with Steve Zissou', 'Time Bandits', 'Cutthroat Island', 'Aliens', 'Guardians of the Galaxy', 'The Hobbit: The Battle of the Five Armies', 'Lions for Lam

In [12]:
l=[]
l.append(simple_recommender("Jack the Giant Slayer"))
print(l)

[{'Elsa & Fred', 'Stripes', 'Space Cowboys', 'The Last Dragon', 'Alive', 'White Squall', 'Miss Congeniality 2: Armed and Fabulous', 'The Abyss', 'Hannah Montana: The Movie', 'Zoom', 'VeggieTales: The Pirates Who Dont Do Anything', 'Risen', 'Justin Bieber: Never Say Never', 'Lost in Space', 'Windtalkers', 'Gangster Squad', 'Firestarter', 'Ice Age: The Meltdown', 'The Proposition', 'Tremors', 'Proof of Life', 'El Mariachi', 'Pandorum', 'Warlock', 'The Other Side of Heaven', 'Mad Max 2: The Road Warrior', 'Tango & Cash', 'The Incredibles', 'A Christmas Story', 'Red Sonja', 'The Boxtrolls', 'Chappie', 'Karachi se Lahore', 'The Hunger Games', 'Avatar', 'Two Brothers', 'The Rugrats Movie', 'Down Terrace', 'Shooter', 'Training Day', 'Crossroads', 'Exit Wounds', 'Run All Night', 'Hollow Man', 'Steel', 'The Core', 'Night at the Museum: Battle of the Smithsonian', 'Time Bandits', 'Cutthroat Island', 'Aliens', 'The Ultimate Gift', 'Guardians of the Galaxy', 'The Hobbit: The Battle of the Five Arm

## Personalized Recommendation and Evaluation

User ratings are incorporated to model individual preferences and generate personalized recommendation scores.

The recommender is trained on user-rating data and evaluated on held-out observations using Precision, Recall, and F1-score.




In [ ]:
from tqdm.notebook import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np
import random


rating_weights = {0: -1, 1: -0.5, 2:0, 3:0, 4:0.5, 5:1}

train_ratings = pd.read_csv("train_ratings.csv")
test_ratings = pd.read_csv("test_ratings.csv")


def calculate_similarity_score(X, Y):
    """
    Return the available content-similarity score between two movies.
    The current knowledge base defines one similarity level.
    """
    query = f"find_sim_1('{X}', '{Y}')"
    return 1 if list(prolog.query(query)) else 0


def train_recommender(ratings, rating_weights, number_of_movies = 10):

    if number_of_movies > len(ratings):
        number_of_movies = len(ratings)


   
    if number_of_movies != -1:
        indexes = random.sample(range(len(ratings)), number_of_movies)



        ratings = ratings.iloc[indexes]


    movie_score = {}
    for row in tqdm(ratings.itertuples(index=True, name='Pandas')):
        movie = clean_text(getattr(row, 'movie_title'))
        rating = getattr(row, 'rating')
        
        similar_movies = simple_recommender(movie)

        for similar_movie in similar_movies:
            similarity_weight = calculate_similarity_score(movie, similar_movie)

            score = rating_weights[int(rating)] * similarity_weight

            if similar_movie not in movie_score:
                movie_score[similar_movie] = score
            else:
                movie_score[similar_movie] += score
    return movie_score



def predict_example(ratings, movie_score):

    real, pred = [], []
    for i, row in enumerate(ratings.itertuples(index=True, name='Pandas')):
        movie = clean_text(getattr(row, 'movie_title'))
        rating = getattr(row, 'rating')

        if movie in movie_score: 
            pred.append(int(movie_score[movie] > 0)) 
            real.append(int(rating > 3))
            
        else: 
            pred.append(0)
            real.append(int(rating > 3))

    return real, pred


def get_metrics(real, pred):
    metrics = {}
    metrics["precision"] = precision_score(real, pred)
    metrics["recall"] = recall_score(real, pred)
    metrics["f1"] = f1_score(real, pred)
    return metrics

In [14]:
metrics = []
for i in range (10):
    
    movie_score = train_recommender(train_ratings, rating_weights)
    real, pred = predict_example(test_ratings, movie_score)
    
    metrics.append(get_metrics(real, pred))


for metric in metrics[0].keys(): 
    print (f"{metric}: {np.mean([m[metric] for m in metrics])}")


0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

...

0it [00:00, ?it/s]

0it [00:00, ?it/s]

c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


0it [00:00, ?it/s]

precision: 0.4580987337781227
recall: 0.7374999999999999
f1: 0.5609800347680471


### Evaluation Strategy

To reduce sensitivity to a particular training subset, the recommender is evaluated across repeated experiments.

Performance is summarized using average Precision, Recall, and F1-score across runs.

In [16]:
metrics = []
for i in range (10):
    movie_score = train_recommender(train_ratings, rating_weights, 10)
    real, pred = predict_example(test_ratings, movie_score)
    metrics.append(get_metrics(real, pred))

for metric in metrics[0].keys():
    print (f"{metric}: {np.mean([m[metric] for m in metrics])}")

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

...

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

precision: 0.535814990385365
recall: 0.7513888888888889
f1: 0.5930858370637802


In [17]:
metrics = []
for i in range (10):
    movie_score = train_recommender(train_ratings, rating_weights, 30)
    real, pred = predict_example(test_ratings, movie_score)
    metrics.append(get_metrics(real, pred))

for metric in metrics[0].keys():
    print (f"{metric}: {np.mean([m[metric] for m in metrics])}")

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

...

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

precision: 0.5117308096210845
recall: 0.9430555555555553
f1: 0.6619242845197473
